Query and visualise: COVID-19 outbreak alerts

Registers the Gold Delta table so it can be queried with real SQL, runs a
couple of queries answering the actual business question, and produces on
chart summarising the result.

In [0]:

# File path for Gold layer external storage location
gold_path = "abfss://gold@ukhsadev2026.dfs.core.windows.net/ukhsa/covid19_outbreak_alerts"

# Registers Gold as a SQL table so %sql / spark.sql() can query it directly
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW covid19_outbreak_alerts AS
SELECT * FROM delta.`{gold_path}`
""")

In [0]:
# Query 1: which local authorities are currently flagged as a spike?
 
spark.sql("""
SELECT geography, current_week_cases, previous_week_cases, percent_change
FROM covid19_outbreak_alerts
WHERE alert_status = 'SPIKE'
ORDER BY percent_change DESC
""").show(50, truncate=False)

In [0]:
# Query 2: top 10 by percentage change regardless of status, gives a sense
# of what's trending even just below the threshold
 
spark.sql("""
SELECT geography, current_week_cases, previous_week_cases, percent_change, alert_status
FROM covid19_outbreak_alerts
ORDER BY percent_change DESC
LIMIT 10
""").show(truncate=False)

In [0]:
# Query 3: overall picture, how many geographies are SPIKE vs BASELINE
 
spark.sql("""
SELECT alert_status, COUNT(*) AS geography_count
FROM covid19_outbreak_alerts
GROUP BY alert_status
""").show()

In [0]:
# Chart: top 10 geographies by percentage change, spikes highlighted
 
import matplotlib.pyplot as plt

top10_df = spark.sql("""
SELECT geography, percent_change, alert_status
FROM covid19_outbreak_alerts
ORDER BY percent_change DESC
LIMIT 10
""").toPandas()
 

 
colors = ["#D85A30" if status == "SPIKE" else "#5DCAA5" for status in top10_df["alert_status"]]
 
plt.figure(figsize=(10, 6))
plt.barh(top10_df["geography"], top10_df["percent_change"], color=colors)
plt.axvline(x=25, color="gray", linestyle="--", linewidth=1, label="25% threshold")
plt.xlabel("Percent change, current week vs previous week")
plt.title("Top 10 local authorities by COVID-19 case change")
plt.gca().invert_yaxis()  # highest change at the top
plt.legend()
plt.tight_layout()
display(plt.gcf())
plt.close()